In [ ]:
# Remove old versions
!pip uninstall -y transformers huggingface-hub tokenizers

# Install required versions
# Install compatible versions for this project
# Main Hugging Face Transformers library
# Download and manage models from Hugging Face Hub
# Optimizes training and inference on CPU/GPU
# Fast tokenizer library
!pip install \
transformers==4.46.3 \
huggingface_hub==0.26.2 \
accelerate==1.1.1 \
tokenizers==0.20.3 \
sentencepiece \
safetensors


In [ ]:
# Import the PyTorch library for tensor operations and GPU support
import torch

# Import Hugging Face classes
# AutoModelForCausalLM = Automatically loads a causal language model (LLM)
# AutoTokenizer = Automatically loads the correct tokenizer for the model
# pipeline =Creates an easy-to-use inference pipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Load the pre-trained Phi-3 model
model = AutoModelForCausalLM.from_pretrained(

    "microsoft/Phi-3-mini-4k-instruct",

    # Automatically selects the available device (GPU/CPU)
    device_map="auto",

    # Automatically chooses the appropriate tensor datatype
    # (float16 on GPU, float32 on CPU)
    torch_dtype="auto",

    # Allows execution of custom model code provided by the model repository
    trust_remote_code=True,
)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(

    # Load the tokenizer associated with the Phi-3 model
    "microsoft/Phi-3-mini-4k-instruct"
)

# Create a text generation pipeline
pipe = pipeline(

    # Specify the task
    "text-generation",

    # Use the previously loaded model
    model=model,

    # Use the loaded tokenizer
    tokenizer=tokenizer,

    # Return only the generated response (not the original prompt)
    return_full_text=False,

    # Maximum number of new tokens the model can generate
    max_new_tokens=500,

    # Disable random sampling for deterministic output
    # Same prompt = Same answer
    do_sample=False,
)

In [ ]:
# Create the prompt as a chat conversation
messages = [
    {
        "role": "user",                     # The sender of the message
        "content": "Create a funny joke about chickens."  # User's prompt
    }
]

# Generate a response from the Phi-3 model
output = pipe(messages)

# Print only the generated response
print(output[0]["generated_text"])

In [ ]:
# Convert the chat messages into the model's expected prompt format
prompt = pipe.tokenizer.apply_chat_template(
    messages,          #Previously given messages role ,content,assistant
    tokenize=False     # Return the prompt as plain text instead of token IDs
)

# Display the formatted prompt
print(prompt)

In [ ]:
# Generate a response using random sampling
output = pipe(
    messages,          # previosuly given messages role,content,assistant
    do_sample=True,    # Enable random sampling instead of always choosing the highest-probability word
    temperature=1      # Set the randomness level (1 = moderate randomness)
)

# Print the generated response
print(output[0]["generated_text"])

In [ ]:
# Generate a response using nucleus (top-p) sampling
output = pipe(
    messages,          # Input conversation
    do_sample=True,    # Enable random sampling
    top_p=1            # Consider 100% of the probability distribution
)

# Print the generated response
print(output[0]["generated_text"])

In [ ]:
# Define the AI's role/persona
persona = "You are an expert in Large Language models. You excel at breaking down complex papers into digestible summaries.\n"

# Tell the model what task to perform
instruction = "Summarize the key findings of the paper provided.\n"

# Provide additional context about the expected summary
context = "Your summary should extract the most crucial points that can help researchers quickly understand the most vital information of the paper.\n"

# Specify the format of the output
data_format = "Create a bullet-point summary that outlines the method. Follow this up with a concise paragraph that encapsulates the main results.\n"

# Specify the target audience
audience = "The summary is designed for busy researchers that quickly need to grasp the newest trends in Large Language Models.\n"

# Specify the writing style/tone
tone = "The tone should be professional and clear.\n"

# Text that needs to be summarized
text = "MY TEXT TO SUMMARIZE"

# Combine the text with a label
data = f"Text to summarize: {text}"

# Build the complete prompt by combining all prompt components
query = (
    persona
    + instruction
    + context
    + data_format
    + audience
    + tone
    + data
)

In [ ]:
# Import the required Hugging Face classes
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Load the pre-trained Phi-3 language model
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="auto",
    torch_dtype="auto",
    trust_remote_code=True
)

# Load the tokenizer for the Phi-3 model
tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    trust_remote_code=True
)

# Create a text-generation pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

# Create a one-shot prompt with one example
one_shot_prompt = [

    # User introduces a new word and asks for an example sentence
    {
        "role": "user",
        "content": "A 'Gigamuru' is a type of Japanese musical instrument. An example of a sentence that uses the word Gigamuru is:"
    },

    # Assistant provides an example sentence
    {
        "role": "assistant",
        "content": "I have a Gigamuru that my uncle gave me as a gift. I love to play it at home."
    },

    # User introduces another new word and expects the model to generate a similar example
    {
        "role": "user",
        "content": "To 'screeg' something is to swing a sword at it. An example of a sentence that uses the word screeg is:"
    }
]

# Convert the chat messages into the model's required prompt format
print(
    tokenizer.apply_chat_template(
        one_shot_prompt,
        tokenize=False
    )
)

In [ ]:
# Generate a response from the model using the one-shot prompt
outputs = pipe(
    one_shot_prompt,
    max_new_tokens=50
)

# Print the generated response
print(outputs[0]["generated_text"])

In [ ]:
# Create a user prompt asking the model to generate a product name and slogan
product_prompt = [
    {
        "role": "user",
        "content": "Create a name and slogan for a chatbot that leverages LLMs."
    }
]

# Generate a response from the model
outputs = pipe(
    product_prompt,
    max_new_tokens=100
)

# Extract the generated text from the model's output
product_description = outputs[0]["generated_text"]

# Display the generated product name and slogan
print(product_description)

In [ ]:
# Create a prompt asking the model to generate a sales pitch
# using the previously generated product description
sales_prompt = [
    {
        "role": "user",
        "content": f"Generate a very short sales pitch for the following product: '{product_description}'"
    }
]

# Generate a response from the model
outputs = pipe(
    sales_prompt,
    max_new_tokens=100
)

# Extract the generated sales pitch
sales_pitch = outputs[0]["generated_text"]

# Display the generated sales pitch
print(sales_pitch)

In [ ]:
# Create a Chain-of-Thought (CoT) prompt with one solved example
cot_prompt = [

    # prompt
    {
        "role": "user",
        "content": "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?"
    },

    # Assistant shows the reasoning step-by-step
    {
        "role": "assistant",
        "content": "Roger started with 5 balls. 2 cans of 3 tennis balls each is 6 tennis balls. 5 + 6 = 11. The answer is 11."
    },

    # prompt to solve using the same reasoning style
    {
        "role": "user",
        "content": "The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have?"
    }
]

# Generate the response from the model
outputs = pipe(
    cot_prompt,
    max_new_tokens=100
)

# Print the generated answer
print(outputs[0]["generated_text"])

In [ ]:
# Create a Zero-Shot Chain-of-Thought prompt
zeroshot_cot_prompt = [

    # Ask the model to solve the problem and explicitly
    {
        "role": "user",
        "content": "The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have? Let's think step-by-step."
    }
]

# Generate a response from the model
outputs = pipe(
    zeroshot_cot_prompt,
    max_new_tokens=100
)

# Print the generated answer
print(outputs[0]["generated_text"])

In [ ]:
# Zero-shot tree-of-thought
zeroshot_tot_prompt = [{
    "role": "user",
    "content": """Imagine three different experts are answering this question. All experts will write down 1 step of their thinking, then share it with the group. Then all experts will go on to the next step, etc. If any expert realizes they're wrong at any point then they leave. The question is 'The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have?' Make sure to discuss the results."""
}]

In [ ]:
# Generate the output
outputs = pipe(zeroshot_tot_prompt,max_new_tokens=100)
print(outputs[0]["generated_text"])

In [ ]:

# Zero-shot learning: Providing no examples
zeroshot_prompt = [
{"role": "user", "content": "Create a character profile for an RPG game in JSON format."}]
# Generate the output
outputs = pipe(zeroshot_prompt)
print(outputs[0]["generated_text"])

In [ ]:
# One-shot learning: Providing an example of the output structure
one_shot_template = """Create a short character profile for an
RPG game. Make sure to only use this format:
{
"description": "A SHORT DESCRIPTION",
"name": "THE CHARACTER'S NAME",
"armor": "ONE PIECE OF ARMOR",
"weapon": "ONE OR MORE WEAPONS"
}
"""
one_shot_prompt = [{"role": "user", "content": one_shot_template}]
# Generate the output
outputs = pipe(one_shot_prompt,max_new_tokens=100)
print(outputs[0]["generated_text"])

In [ ]:
# Import the garbage collection module
# The garbage collector automatically removes objects that are no longer being used, helping to free RAM.
import gc

# Import the PyTorch library
import torch

# Delete the model, tokenizer, and pipeline objects from memory
del model, tokenizer, pipe

# Run Python's garbage collector to free unused memory
gc.collect()

# Clear the GPU cache used by PyTorch
torch.cuda.empty_cache()

In [ ]:
# Install the llama-cpp-python library for running GGUF models locally
# This library allows  to run GGUF models locally without using the Hugging Face Transformers library.
# GGUF (GPT-Generated Unified Format) is a model file format designed for efficient local inference.
!pip install llama-cpp-python

# Import the Llama class from the llama_cpp library
from llama_cpp.llama import Llama

# Load the Phi-3 GGUF model
llm = Llama.from_pretrained(

    # Hugging Face repository containing the GGUF model
    repo_id="microsoft/Phi-3-mini-4k-instruct-gguf",

    # Load the FP16 GGUF model file
    filename="*fp16.gguf",

    # Use all available GPU layers for faster inference
    n_gpu_layers=-1,

    # Set the maximum context window to 2048 tokens
    n_ctx=2048,

    # Disable detailed loading logs
    verbose=False
)

In [ ]:
# Generate a chat completion from the loaded LLM
output = llm.create_chat_completion(

    # Provide the conversation (prompt) to the model
    messages=[
        {
            "role": "user",
            "content": "Create a warrior for an RPG in JSON format."
        },
    ],

    # Force the model to return the output as a valid JSON object
    response_format={"type": "json_object"},

    # Set temperature to 0 for deterministic and consistent output
    temperature=0,

# Extract only the generated JSON response from the model's output
)['choices'][0]['message']["content"]

In [ ]:
# Import the built-in JSON library
import json

# Convert the JSON string into a Python dictionary and  then format it with proper indentation
json_output = json.dumps(
    json.loads(output),
    indent=4
)

# Display the formatted JSON
print(json_output)